# Ablation study

In [ ]:
# Colab: mount Google Drive (skip if already mounted / running locally)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Not on Colab or already mounted:", e)

In [ ]:
CORE_NB = "/content/drive/MyDrive/Depression_FL_Project/notebooks/1_FL_Fusion_Core.ipynb"


RUN_VARIANTS = ["fedproto", "fedtgp"]     # subset of ["fedproto", "fedtgp"]


QUICK_TEST   = False


# Set of ablations to run. Comment out any you don't want.
ABLATIONS_TO_RUN = [
    "full (all components)",
    "- ETF init",
    "- adaptive gating (uniform g=1)",
    "- gate adaptation (freeze at init)",
    "- focal loss",
    "- classifier head",
    "- compactness",
    "- reliability weighting",
]

In [ ]:
import json, os

def _strip_magics(src):
    "Drop IPython/shell lines (!pip, %magic, get_ipython) so plain exec() can run the cell."
    keep = []
    for ln in src.splitlines():
        s = ln.lstrip()
        if s.startswith("!") or s.startswith("%") or s.startswith("get_ipython("):
            keep.append("# [stripped magic] " + ln)
        else:
            keep.append(ln)
    return "\n".join(keep)

def _load_core_setup(core_nb_path, run_cells):
    assert os.path.exists(core_nb_path), f"Core notebook not found: {core_nb_path}\n-> edit CORE_NB."
    nb = json.load(open(core_nb_path))
    code_cells = [''.join(c['source']) for c in nb['cells'] if c['cell_type'] == 'code']
    print(f"core notebook has {len(code_cells)} code cells; running setup cells {run_cells} ...\n")
    for idx in run_cells:
        print(f"===== core code cell {idx} =====")
        exec(compile(_strip_magics(code_cells[idx]), f'<core_cell_{idx}>', 'exec'), globals())
    print("\n[setup done]")

_load_core_setup(CORE_NB, [1, 2, 3, 4, 5, 6, 7, 8, 9])

# sanity: the objects the ablation depends on must now exist
for _name in ["tdata", "in_dims", "run_fl_training", "predict_labels",
              "best_threshold_on_val", "standalone_results", "N_SEEDS", "LR_HEAD"]:
    assert _name in globals(), f"missing '{_name}' -- did the core setup run fully?"
print("required objects present:", ["tdata", "in_dims", "run_fl_training", "standalone_results"])

In [ ]:
import numpy as np, pandas as pd, torch, copy
from sklearn.metrics import f1_score

_ABLATION_KNOBS = ["USE_ETF_INIT", "USE_FOCAL_LOSS", "USE_CLASSIFIER_HEAD", "PREDICT_WITH",
                   "USE_COMPACTNESS", "USE_CLIENT_RELIABILITY_WEIGHTING",
                   "GATE_INIT", "GATE_ADAPT_MOMENTUM"]

def _snapshot():
    return {k: copy.deepcopy(globals()[k]) for k in _ABLATION_KNOBS if k in globals()}

def _restore(snap):
    for k, v in snap.items():
        globals()[k] = copy.deepcopy(v)

def _ablation_overrides(full_gate_init):
    uniform_gate = {k: 1.0 for k in full_gate_init}
    return {
        "full (all components)":              {},
        "- ETF init":                         {"USE_ETF_INIT": False},
        "- adaptive gating (uniform g=1)":    {"GATE_INIT": uniform_gate, "GATE_ADAPT_MOMENTUM": 1.0},
        "- gate adaptation (freeze at init)": {"GATE_ADAPT_MOMENTUM": 1.0},
        "- focal loss":                       {"USE_FOCAL_LOSS": False},
        "- classifier head":                  {"USE_CLASSIFIER_HEAD": False, "PREDICT_WITH": "prototype"},
        "- compactness":                      {"USE_COMPACTNESS": False},
        "- reliability weighting":            {"USE_CLIENT_RELIABILITY_WEIGHTING": False},
    }

def _eval_fl(heads, protos):
    "Mirror the core Table 2/3 evaluation: per-silo macro-F1 with val-tuned thresholds, plus overall."
    per_silo, yt_all, yp_all = {}, [], []
    for silo in tdata:
        heads[silo].eval()
        thr = best_threshold_on_val(heads[silo], tdata[silo]["val"][0], tdata[silo]["val"][1])
        with torch.no_grad():
            emb = heads[silo](tdata[silo]["test"][0])
        pred = predict_labels(heads[silo], emb, protos, pos_threshold=thr)
        yt = tdata[silo]["test"][1].numpy(); yp = pred.cpu().numpy()
        per_silo[silo] = f1_score(yt, yp, average="macro", zero_division=0)
        yt_all.append(yt); yp_all.append(yp)
    unweighted      = float(np.mean(list(per_silo.values())))
    yt_all, yp_all  = np.concatenate(yt_all), np.concatenate(yp_all)
    sample_weighted = f1_score(yt_all, yp_all, average="macro", zero_division=0)
    return per_silo, unweighted, sample_weighted

def run_ablation(variant="fedproto", seeds=None, which=None):
    use_tgp = (variant == "fedtgp")
    n_seeds = seeds if seeds is not None else N_SEEDS
    full_gate_init = dict(globals()["GATE_INIT"])
    all_overrides  = _ablation_overrides(full_gate_init)
    which = list(all_overrides.keys()) if which is None else which
    snap, records = _snapshot(), []
    try:
        for name in which:
            overrides = all_overrides[name]
            _restore(snap)                        # reset to full config
            for k, v in overrides.items():        # apply this single ablation
                globals()[k] = v
            per_silo_seeds = {s: [] for s in tdata}
            unw, sw = [], []
            for seed in range(n_seeds):
                heads, protos, best_round, hist = run_fl_training(
                    tdata, in_dims, seed, use_trainable_global=use_tgp, lr=LR_HEAD)
                ps, u, s = _eval_fl(heads, protos)
                for silo in tdata:
                    per_silo_seeds[silo].append(ps[silo])
                unw.append(u); sw.append(s)
            rec = {"ablation": name, "variant": variant}
            for silo in tdata:
                rec[silo]          = float(np.mean(per_silo_seeds[silo]))
                rec[silo + "_std"] = float(np.std(per_silo_seeds[silo]))
            rec["overall_unweighted"]      = float(np.mean(unw))
            rec["overall_sample_weighted"] = float(np.mean(sw))
            records.append(rec)
            print(f"[{variant}] {name:34s} " +
                  " ".join(f"{s}={rec[s]:.3f}" for s in tdata) +
                  f" | unw={rec['overall_unweighted']:.3f} sw={rec['overall_sample_weighted']:.3f}")
    finally:
        _restore(snap)                            # ALWAYS restore the full config
    return pd.DataFrame(records)

print("ablation harness ready.")

In [ ]:
_variants = ["fedproto"] if QUICK_TEST else RUN_VARIANTS
_seeds    = QUICK_SEEDS if QUICK_TEST else N_SEEDS

print(f"Running ablation | variants={_variants} | seeds={_seeds} | "
      f"standalone floor per silo: " +
      ", ".join(f"{s}={standalone_results[s][0]:.3f}" for s in tdata) + "\n")

ablation_tables = {}
for _v in _variants:
    print(f"\n================= variant: {_v} =================")
    ablation_tables[_v] = run_ablation(_v, seeds=_seeds, which=ABLATIONS_TO_RUN)

ablation_df = pd.concat(ablation_tables.values(), ignore_index=True)
ablation_df

In [ ]:
def deltas_vs_full(df_variant):
    silos = [c for c in df_variant.columns if c in tdata]
    metrics = silos + ["overall_unweighted", "overall_sample_weighted"]
    full = df_variant[df_variant.ablation == "full (all components)"].iloc[0]
    rows = []
    for _, r in df_variant.iterrows():
        if r.ablation == "full (all components)":
            continue
        rows.append({"variant": r.variant, "removed": r.ablation.lstrip("- "),
                     **{m: round(float(full[m] - r[m]), 4) for m in metrics}})
    return pd.DataFrame(rows)

delta_frames = {v: deltas_vs_full(t) for v, t in ablation_tables.items()}
for v, d in delta_frames.items():
    print(f"\n===== {v}: macro-F1 drop when each component is removed (higher = more important) =====")
    print(d.to_string(index=False))

delta_all = pd.concat(delta_frames.values(), ignore_index=True)
delta_all

In [ ]:
# Save tables and a grouped bar chart to the core OUT directory.
import os, matplotlib.pyplot as plt

_outdir = globals().get("OUT", ".")
os.makedirs(_outdir, exist_ok=True)
ablation_df.to_csv(f"{_outdir}/ablation_results.csv", index=False)
delta_all.to_csv(f"{_outdir}/ablation_deltas.csv", index=False)
print("saved:", f"{_outdir}/ablation_results.csv", "and ablation_deltas.csv")

for _v, _t in ablation_tables.items():
    silos = [c for c in _t.columns if c in tdata]
    labels = list(_t.ablation)
    x = np.arange(len(labels)); width = 0.8 / max(len(silos), 1)
    fig, ax = plt.subplots(figsize=(max(9, 1.3 * len(labels)), 5))
    for i, s in enumerate(silos):
        ax.bar(x + i * width, _t[s], width, yerr=_t[s + "_std"], capsize=3, label=s)
    # standalone floors as dashed lines
    for s in silos:
        ax.axhline(standalone_results[s][0], ls="--", lw=0.8, alpha=0.5)
    ax.set_xticks(x + width * (len(silos) - 1) / 2)
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_ylabel("Test macro-F1"); ax.set_title(f"Component ablation ({_v})")
    ax.legend(); ax.set_ylim(0, 1); fig.tight_layout()
    fig.savefig(f"{_outdir}/ablation_{_v}.png", dpi=200)
    plt.show()
    print("saved figure:", f"{_outdir}/ablation_{_v}.png")

## Shared-space visualization (before vs after alignment)

In [ ]:
# Uses the FULL config (restored by the harness above).
import torch, numpy as np

def _project_all(heads):
    Z, Y, S = [], [], []
    for silo in tdata:
        heads[silo].eval()
        with torch.no_grad():
            z = heads[silo](tdata[silo]["test"][0]).cpu().numpy()   # (n, 512), L2-normalized
        Z.append(z); Y.append(tdata[silo]["test"][1].numpy()); S += [silo] * len(z)
    return np.concatenate(Z), np.concatenate(Y), np.array(S)

torch.manual_seed(0)
heads_before   = {silo: ProjectionHead(in_dims[silo]) for silo in tdata}
protos_before  = _init_proto_bank(N_CLASSES, SHARED_DIM, use_etf=USE_ETF_INIT)
Zb, Yb, Sb     = _project_all(heads_before)

# AFTER: one full FL run (vanilla FedProto, seed 0)
heads_after, protos_after, _, _ = run_fl_training(
    tdata, in_dims, 0, use_trainable_global=False, lr=LR_HEAD)
Za, Ya, Sa = _project_all(heads_after)

proto_mat_before = np.stack([protos_before[c].cpu().numpy() for c in range(N_CLASSES)])
proto_mat_after  = np.stack([protos_after[c].cpu().numpy()  for c in range(N_CLASSES)])
print("before:", Zb.shape, "after:", Za.shape, "| classes:", N_CLASSES)

In [ ]:
# 2-D embedding (UMAP if available, else t-SNE) and side-by-side plot.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

USE_UMAP = True   # Falls back to t-SNE automatically if UMAP is unavailable

def reduce_2d(Z, protos, seed=0):
    """Fit on points + prototypes together so prototypes share the map."""
    X = np.vstack([Z, protos])
    n = len(Z)

    reducer = None
    if USE_UMAP:
        try:
            import umap
            reducer = umap.UMAP(
                n_neighbors=min(15, n - 1),
                min_dist=0.1,
                metric="cosine",
                random_state=seed
            )
        except Exception as e:
            print("UMAP unavailable, using t-SNE:", e)

    if reducer is None:
        from sklearn.manifold import TSNE
        reducer = TSNE(
            n_components=2,
            perplexity=min(15, max(5, n // 4)),
            init="pca",
            metric="cosine",
            random_state=seed
        )

    E = reducer.fit_transform(X)
    return E[:n], E[n:]


# Reduce dimensionality
pts_b, pr_b = reduce_2d(Zb, proto_mat_before)
pts_a, pr_a = reduce_2d(Za, proto_mat_after)

# Colors and markers
class_colors = ["#1f77b4", "#d62728", "#2ca02c"][:N_CLASSES]
silo_markers = {s: m for s, m in zip(tdata.keys(), ["o", "s", "^", "D", "v"])}


def panel(ax, pts, pr, Y, S, title):
    for silo in tdata:
        for c in range(N_CLASSES):
            mask = (S == silo) & (Y == c)
            if mask.any():
                ax.scatter(
                    pts[mask, 0],
                    pts[mask, 1],
                    c=class_colors[c],
                    marker=silo_markers[silo],
                    s=40,
                    alpha=0.8,
                    edgecolors="k",
                    linewidths=0.3
                )

    # Plot prototypes
    for c in range(N_CLASSES):
        ax.scatter(
            pr[c, 0],
            pr[c, 1],
            c=class_colors[c],
            marker="*",
            s=420,
            edgecolors="k",
            linewidths=1.2,
            zorder=5
        )

    ax.set_title(title, fontsize=14)
    ax.set_xticks([])
    ax.set_yticks([])


# Create figure
fig, axes = plt.subplots(1, 2, figsize=(14, 6.8))

panel(axes[0], pts_b, pr_b, Yb, Sb,
      "Before alignment (untrained heads)")

panel(axes[1], pts_a, pr_a, Ya, Sa,
      "After FL alignment")

# Legend handles
class_handles = [
    Line2D([0], [0],
           marker="o",
           linestyle="",
           markerfacecolor=class_colors[c],
           markeredgecolor="k",
           label=f"Class {c}")
    for c in range(N_CLASSES)
]

silo_handles = [
    Line2D([0], [0],
           marker=silo_markers[s],
           linestyle="",
           markerfacecolor="gray",
           markeredgecolor="k",
           label=s.capitalize())
    for s in tdata
]

proto_handle = [
    Line2D([0], [0],
           marker="*",
           linestyle="",
           markerfacecolor="gold",
           markeredgecolor="k",
           markersize=15,
           label="Global Prototype")
]

# Figure legend (placed BELOW the plots)
fig.legend(
    handles=class_handles + silo_handles + proto_handle,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.02),
    ncol=len(class_handles) + len(silo_handles) + 1,
    frameon=False,
    fontsize=11
)

# Overall title
fig.suptitle(
    "Shared 512-d Space: Same Class Across Different Silos",
    fontsize=16,
    y=0.98
)

# Leave room at bottom for legend
fig.tight_layout(rect=[0, 0.10, 1, 0.94])

# Save figure
_outdir = globals().get("OUT", ".")
fig.savefig(
    f"{_outdir}/shared_space_tsne_before_after.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", f"{_outdir}/shared_space_tsne_before_after.png")